In [1]:
import torch
import torch.nn as nn

_ = torch.manual_seed(26)
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

In [2]:
batch_size = 64
sequence_len = 64
embedding_size = 32
hidden_size = 256
steps = 5000

In [3]:
# input & output initialization
with open(file="inputs-shakespeare.txt", mode="r", encoding="utf-8") as f:
    data = f.read()

chars = sorted(set(data))
vocab_size = len(chars)
ctoi = {char: i for i, char in enumerate(iterable=chars)}
itoc = dict(enumerate(iterable=chars))

split = int(len(data) * 0.9)
train, val = data[:split], data[split:]

# train stays on cpu (unfold is a view, moving it would copy every window), batches move to device
x_train = torch.tensor([ctoi[ch] for ch in train[:-1]], dtype=torch.long).unfold(dimension=0, size=sequence_len, step=1)
y_train = torch.tensor([ctoi[ch] for ch in train[1:]], dtype=torch.long).unfold(dimension=0, size=sequence_len, step=1)

x_val = torch.tensor([ctoi[ch] for ch in val[:-1]], dtype=torch.long).unfold(dimension=0, size=sequence_len, step=sequence_len).to(device=device)
y_val = torch.tensor([ctoi[ch] for ch in val[1:]], dtype=torch.long).unfold(dimension=0, size=sequence_len, step=sequence_len).to(device=device)

In [4]:
# module declaration
class CausalMean(nn.Module):
    def __init__(self, sequence_len):
        super().__init__()
        tril = torch.tril(input=torch.ones(size=(sequence_len, sequence_len), dtype=torch.bool))
        self.register_buffer(name="mask", tensor=tril, persistent=False)

    def forward(self, input):
        T = input.size(dim=1)
        scores = torch.zeros(size=(T, T), device=input.device)
        scores = scores.masked_fill(mask=~self.mask[:T, :T], value=float("-inf"))
        weights = torch.softmax(input=scores, dim=-1)
        return weights @ input

In [5]:
# model declaration
class BagOfWords(nn.Module):
    def __init__(self, vocab_size, embedding_size, sequence_len, hidden_size):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Embedding(num_embeddings=vocab_size, embedding_dim=embedding_size),
            CausalMean(sequence_len=sequence_len),
            nn.Linear(in_features=embedding_size, out_features=hidden_size),
            nn.ReLU(),
            nn.Linear(in_features=hidden_size, out_features=vocab_size),
        )

    def forward(self, input):
        return self.layers(input=input)

In [6]:
# model initialization
model = BagOfWords(vocab_size=vocab_size, embedding_size=embedding_size, sequence_len=sequence_len, hidden_size=hidden_size).to(device=device)
print(f"parameters: {sum(param.numel() for param in model.parameters())}")
# optimizer initialization
optimizer = torch.optim.AdamW(params=model.parameters(), lr=3e-3)

parameters: 27233


In [7]:
# training
for step in range(steps):
    optimizer.zero_grad()

    batch = torch.randint(low=0, high=len(x_train), size=(batch_size,))
    x_batch = x_train[batch].to(device=device)
    y_batch = y_train[batch].to(device=device)
    train_logits = model(input=x_batch)
    train_loss = nn.functional.cross_entropy(input=train_logits.view(-1, vocab_size), target=y_batch.view(-1))

    if step % 1000 == 0 or step == steps - 1:
        with torch.no_grad():
            val_logits = model(input=x_val)
            val_loss = nn.functional.cross_entropy(input=val_logits.view(-1, vocab_size), target=y_val.view(-1))
            print(f"Step {step}: train loss = {train_loss.item()}, val loss = {val_loss.item()}")

    train_loss.backward()
    optimizer.step()

Step 0: train loss = 4.17216682434082, val loss = 4.173493385314941
Step 1000: train loss = 3.0138981342315674, val loss = 3.0591413974761963
Step 2000: train loss = 2.9847283363342285, val loss = 3.028728485107422
Step 3000: train loss = 2.971952438354492, val loss = 3.012568712234497
Step 4000: train loss = 2.9784998893737793, val loss = 3.014688730239868
Step 4999: train loss = 2.974222421646118, val loss = 3.005603551864624


In [8]:
# prediction
with torch.inference_mode():
    output = ['R', 'O', 'M', 'E', 'O', ':', '\n']
    max_output_length = 1000

    i = torch.tensor([[ctoi[ch] for ch in output]], dtype=torch.long, device=device)

    while len(output) < max_output_length:
        logits = model(input=i)

        P = torch.softmax(input=logits[0, -1, :], dim=0)
        j = torch.multinomial(input=P, num_samples=1, replacement=True).item()
        output.append(itoc[j])

        i = torch.tensor([[ctoi[ch] for ch in output[-sequence_len:]]], dtype=torch.long, device=device)

In [9]:
print(''.join(output))

ROMEO:
I
hOe,,  ly  spoussa tstyfpleo ,hl a tdaadal  vtore leoypnelld;uoaoe,.
 o a
TCh inR
 EcemnAtLIE N:
rfladynntwLdbeddn, :nrdwooh  
notIIfeiinho  iewoen
  u;e w bu w wb dtnof ceua ghutnAtnoaeusin  thsgyhhPosd aa'paywgeut nrs r eg awr,udl,fiah hmeo, n w nratios.eogn 
f Fe
Kle' etaHh aiye utIrsJ  hkDglIhElMIA EOIoN:rgg.r 
seNa wgtVir
esidsv
ypt  d.ig: 
,.wee

eyCeh, Aoo uu w ea
MkIOECMdJBR,VmUFUtMEyoIYAL eI:
  lv Ntesa lrrI :akhii.'e
  me er
aro tliruyhpovsfgao drev
Ignutthu W  i ,ehfinWe Y ynnoaycu'an ldh, :g c,'aetrh?woabo e
d o'lotr
s

erfOpIy ki f  oT necrmosWwe a nokrlOhEtNuRKVWxTeOw  oKYlfsn:oee
 afe 
nwt uhre e od!el:ent'h rwgodf
;h ibh ehht,oFi  SseTa ;kro 
eudiaadom Silssshyh  nbtIh  ftreborfua yotrlty    eeon u e ocbrircng:nulam
o lthewk  hheoaroasmsynenenauekdif t eluir   shn,n
 g iys  htE'dhae st fdchkfnaoo r itigupeeoblbdttudnghICi  l a aeygamn att
d s? sle ioutpelso n'y nheoa sl,  aspumgTsatngn lwwi eoso c oeufisy?v n
   iehalLntiawsoihqldthenolonglrdll l mogs?atfda
 ,
